In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import io
import math
import random
from PIL import Image
import glob
import pandas as pd
from matplotlib.patches import Rectangle
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import colorcet as cc
import seaborn as sns
from matplotlib.colors import ListedColormap

In [2]:
def show_img(img, title=None):
    img = img.squeeze()

    plt.imshow(img, cmap='gray')
    if title != None:
        plt.title(title)
    plt.axis('off')


In [ ]:
import os
import json
import pandas as pd

root = "msgs"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if 'dir' in k.lower() or 'split' in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace('experiments/', '').replace('/results', '')
        records.append(record)

df = pd.DataFrame(records)

df.rename(columns={"test_time_training_accuracy": "test_time_mutual_accuracy"}, inplace=True)
df.rename(columns={"test_time_self_play_accuracy": "test_time_self_accuracy"}, inplace=True)

df.loc[(df["num_iterations"] == 0), "test_time_mode"] = "-"

In [4]:
def filter_df(filters, df=df, sort_by=["message_length", "message_length_tt", "learning_rate_tt", "num_iterations"]):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if (type(value) == str) and value.startswith('>'):
                mask |= (df[key] > value.split('>')[1])
            if value == "!None":
                mask |= (df[key] != "None")
            else:
                mask |= (df[key] == value)
        idx &= mask 

    return df[idx].sort_values(by=sort_by).reset_index(drop=True)    

In [5]:
adapt_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

In [6]:
def load(df):
    assert len(df) == 1
    
    return  torch.load(df.loc[0, "path"] + "/messages.pt")

In [ ]:
def to_2d(log, method="tsne", representation="receiver_representations"):
    reps = log[representation].detach().cpu()

    msg_vecs = reps.reshape(reps.size(0), -1).numpy()

    if method == "pca":
        X2 = PCA(n_components=2).fit_transform(msg_vecs)
    elif method == "tsne":
        X2 = TSNE(n_components=2, init="pca", learning_rate="auto").fit_transform(msg_vecs)
    else:
        raise ValueError("method must be 'pca' or 'tsne'")
    
    return X2


def plot_2d(X2, block_size=100, color="glasbey_dark",
            save_path="tsne_plot", show=False):

    plt.figure(figsize=(7, 6), dpi=300)

    num_classes = int(np.ceil(len(X2) / block_size))

    if color == "glasbey_dark":
        cmap = ListedColormap(cc.glasbey_dark[:num_classes])
    else:
        cmap = plt.cm.get_cmap("tab20", num_classes)

    group_ids = np.arange(len(X2)) // block_size

    plt.scatter(
        X2[:, 0],
        X2[:, 1],
        c=group_ids,
        cmap=cmap,
        s=12,
        alpha=0.85,
        linewidths=0
    )

    # remove axes for embedding visualization
    plt.xticks([])
    plt.yticks([])

    # remove frame
    for spine in plt.gca().spines.values():
        spine.set_visible(False)

    plt.gca().set_aspect("equal", adjustable="box")

    plt.tight_layout()

    # save publication quality figure
    plt.savefig(
        f"../assets/{save_path}.pdf",
        dpi=300,
        bbox_inches=None,
        format="pdf"
    )

    if show:
        plt.show()
    else:
        plt.close ()

In [ ]:
def plot_message_examples(
    images,
    messages_before,
    messages_after,
    n=8,
    seed=1,
    save_path="message_example",
    dataset="imagenet",
    cmap=None,
    show=False,
    width=2
):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(images), size=n, replace=False)

    fig, axes = plt.subplots(
        n,
        2,
        figsize=(width, 0.9 * n),
        dpi=300,
        gridspec_kw={"width_ratios": [1.2, 1.2]},
    )

    if n == 1:
        axes = axes[None, :]

    for row, i in enumerate(idx):
        ax = axes[row, 0]

        if dataset == "imagenet":
            img = get_imgnet_img(images[i])
        else:
            img = images[i]
        
        ax.imshow(img, cmap=cmap, aspect="equal")
        ax.axis("off")

        before = str(messages_before[i].tolist()).replace(", ", " ").replace("[", "(").replace("]", ")")
        after = str(messages_after[i].tolist()).replace(", ", " ").replace("[", "(").replace("]", ")")

        msg = f"{before} → {after}"

        ax = axes[row, 1]
        ax.text(
            0.0,
            0.5,
            msg,
            ha="left",
            va="center",
            fontsize=7,
            family="monospace",
            transform=ax.transAxes,
        )
        ax.axis("off")

    plt.subplots_adjust(
        left=0.02,
        right=0.99,
        top=0.99,
        bottom=0.01,
        wspace=0.12,
        hspace=0.1,
    )

    plt.savefig(f"../assets/messages/{save_path}.pdf", bbox_inches="tight", pad_inches=0.05)
    
    if show:
        plt.show()
    else:
        plt.close()
        


# Imagenet

In [125]:
files = sorted(glob.glob("/home/shared/data/imagenet/validation-*.parquet"))

df_imagenet = pd.read_parquet(files)


def get_imgnet_img(idx, size=(224, 224)):
    img_bytes = df_imagenet.loc[int(idx), 'image']['bytes']
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    img = img.resize(size, Image.Resampling.LANCZOS)
    return img

def show_imgnet(idx, df=df_imagenet, title=None):
    img_bytes = df.loc[int(idx), 'image']['bytes']
    img = Image.open(io.BytesIO(img_bytes))

    plt.imshow(img)
    plt.axis("off")
    if title != None:
        plt.title(title)

## VQEL

In [126]:
res = filter_df({
    "dataset": "imagenet",
    "test_time_mode": "-",
    "message_length_tt": 4
})

res[adapt_cols]

In [127]:
vqel_l4 = load(res)

In [128]:
res = filter_df({
    "dataset": "imagenet",
    "test_time_mode": "-",
    "message_length_tt": 5
})
res[adapt_cols]

In [129]:
vqel_l5 = load(res)

## Dataset Adaptation

In [130]:
res = filter_df({
    "dataset": "imagenet",
    "test_time_mode": "dataset_adaptation",
})

res[adapt_cols]

In [131]:
dataset_adapt = load(res)

## Plot

### Receiver Representation

In [81]:
# x = to_2d(vqel_l4)
# plot_2d(x, block_size=32)

In [82]:
# x = to_2d(vqel_l5)
# plot_2d(x)

In [83]:
# x = to_2d(dataset_adapt)
# plot_2d(x)

### Sender Representation

In [84]:
# x = to_2d(vqel_l4, representation="sender_representations")
# plot_2d(x)

In [85]:
# x = to_2d(vqel_l5, representation="sender_representations")
# plot_2d(x)

In [86]:
# x = to_2d(dataset_adapt, representation="sender_representations")
# plot_2d(x)

### Words

In [25]:
x = to_2d(vqel_l4, representation="words")
plot_2d(x, save_path="tsne/imagenet_vqel_l4")

In [26]:
x = to_2d(vqel_l5, representation="words")
plot_2d(x, save_path="tsne/imagenet_vqel_l5")

In [27]:
x = to_2d(dataset_adapt, representation="sender_representations")
plot_2d(x, save_path="tsne/imagenet_dataset_adapt")

### Samples

In [ ]:

indices = dataset_adapt["indices"]
words_before = vqel_l4["words"]
words_after = dataset_adapt["words"]

plot_message_examples(
    indices,
    words_before,
    words_after,
    n=5,
    seed=0,
    show=True,
    save_path="imagenet"
)

# MNIST

## VQEL

In [148]:
res = filter_df({
    "dataset": "mnist1",
    "test_time_mode": "-",
    "message_length_tt": 4,
})

res[adapt_cols]

In [149]:
vqel_l4 = load(res)

In [150]:
res = filter_df({
    "dataset": "mnist1",
    "test_time_mode": "-",
    "message_length_tt": 10,
})
res[adapt_cols]

In [151]:
vqel_l10 = load(res)

## Dataset Adaptation

In [152]:
res = filter_df({
    "dataset": "mnist1",
    "test_time_mode": "dataset_adaptation",
})
res[adapt_cols]

In [153]:
dataset_adapt = load(res)

## Plot

### Receiver Representaion

In [107]:
# x = to_2d(vqel_l4)
# plot_2d(x)

In [108]:
# x = to_2d(vqel_l10)
# plot_2d(x)

In [109]:
# x = to_2d(dataset_adapt)
# plot_2d(x)

### Sencer Representation

In [110]:
# x = to_2d(vqel_l4, representation="sender_representations")
# plot_2d(x)

In [111]:
# x = to_2d(vqel_l10, representation="sender_representations")
# plot_2d(x)

In [112]:
# x = to_2d(dataset_adapt, representation="sender_representations")
# plot_2d(x)

### Words

In [15]:
x = to_2d(vqel_l4, representation="words")
plot_2d(x, save_path="tsne/mnist_vqel_l4")

In [16]:
x = to_2d(vqel_l10, representation="words")
plot_2d(x, save_path="tsne/mnist_vqel_l10")

In [17]:
x = to_2d(dataset_adapt, representation="words")
plot_2d(x, save_path="tsne/mnist_dataset_adapt")

### Samples

In [ ]:
indices = dataset_adapt["imgs"]
words_before = vqel_l4["words"]
words_after = dataset_adapt["words"]

plot_message_examples(
    indices,
    words_before,
    words_after,
    n=5,
    seed=0,
    dataset="mnist",
    cmap="gray",
    save_path="mnist",
    width=3.5,
)